In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import torch
from scsims import SIMS

/Users/zoezabetian/miniconda3/envs/sims_env_310/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Load data

In [2]:
adata_luo = sc.read_h5ad('/Users/zoezabetian/Desktop/M-R-lab/leo_project/luo_files/Luo2021nature_new_labels.h5ad')
adata_wendering = sc.read_h5ad('/Users/zoezabetian/Desktop/M-R-lab/leo_project/wend_files/Wend2024sciadv_two_labels.h5ad')
adata_leo_treg = sc.read_h5ad('/Users/zoezabetian/Desktop/M-R-lab/immune-sims/leo_data/fer_cd4_treg_ngs_final.h5ad')
adata_leo_teff = sc.read_h5ad('/Users/zoezabetian/Desktop/M-R-lab/immune-sims/leo_data/fer_cd4_teff_ngs_final.h5ad')

/Users/zoezabetian/miniconda3/envs/sims_env_310/lib/python3.9/site-packages/anndata/_core/anndata.py:1832: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/zoezabetian/miniconda3/envs/sims_env_310/lib/python3.9/site-packages/anndata/compat/__init__.py:229: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/Users/zoezabetian/miniconda3/envs/sims_env_310/lib/python3.9/site-packages/anndata/compat/__init__.py:229: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


2. Create the same labels

luo:

In [5]:
adata_luo.obs.head()
unique_cell_types = adata_luo.obs['treg'].unique()
print(unique_cell_types)

['effector', 'naive']
Categories (2, object): ['effector', 'naive']


In [7]:
# update Luo dataset labels in the 'treg' column
adata_luo.obs['treg'] = adata_luo.obs['treg'].replace({
    'effector': 'Teffector',
    'naive': 'Treg'
})
unique_cell_types = adata_luo.obs['treg'].unique()
print(unique_cell_types)

['Teffector', 'Treg']
Categories (2, object): ['Teffector', 'Treg']


In [11]:
cell_type_counts = adata_luo.obs['treg'].value_counts()
print("\nCell type distribution:")
print(cell_type_counts)
adata_luo.n_obs


Cell type distribution:
treg
Treg         3658
Teffector    3333
Name: count, dtype: int64


6991

wend:

In [6]:
print(adata_wendering.obs['treg'].unique())

['Teffector', 'Treg']
Categories (2, object): ['Teffector', 'Treg']


In [9]:
cell_type_counts = adata_wendering.obs['treg'].value_counts()
print("\nCell type distribution:")
print(cell_type_counts)



Cell type distribution:
treg
Teffector    11081
Treg          8305
Name: count, dtype: int64


leo data: 
- remove cols, add 'treg' column to both datasets, label 'Treg' for treg set and 'Teff' for teff set

In [17]:
print(adata_leo_teff.obs.columns)
unique = adata_leo_teff.obs['tcell_type'].unique()
print(unique)

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nUMI', 'nGene', 'sample',
       'pMito', 'patient', 'act_tcell', 'tcell_type', 'activation',
       'mito_filter_out', 'numi_out', 'all_out', 'integrated_snn_res.0.1',
       'integrated_snn_res.0.2', 'integrated_snn_res.0.3',
       'integrated_snn_res.0.4', 'integrated_snn_res.0.5',
       'integrated_snn_res.0.6', 'integrated_snn_res.0.7',
       'integrated_snn_res.0.8', 'integrated_snn_res.0.9',
       'integrated_snn_res.1', 'integrated_snn_res.1.1',
       'integrated_snn_res.1.2', 'seurat_clusters', 'scDblFinder.class',
       'garnett_celltypes', 'activation_status', 'Cell', 'Cluster1',
       'Cluster2', 'down_treg', 'up_treg', 'clusters', 'S.Score', 'G2M.Score',
       'Phase', 'treg_score', 'RNA_snn_res.0.1', 'RNA_snn_res.0.2',
       'RNA_snn_res.0.3', 'RNA_snn_res.0.4', 'RNA_snn_res.0.5',
       'RNA_snn_res.0.6', 'RNA_snn_res.0.7', 'RNA_snn_res.0.8',
       'RNA_snn_res.0.9', 'RNA_snn_res.1', 'RNA_snn_res.1.1',
       

In [20]:
# adata_leo_teff.obs = adata_leo_teff.obs[['tcell_type']]
# adata_leo_teff.obs.rename(columns={'tcell_type': 'treg'}, inplace=True)
print(adata_leo_teff.obs.columns)
cell_type_counts = adata_leo_teff.obs['treg'].count()
print(cell_type_counts)

Index(['treg'], dtype='object')
45955


In [21]:
print(adata_leo_treg.obs.columns)
unique = adata_leo_treg.obs['tcell_type'].unique()
print(unique)

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nUMI', 'nGene', 'sample',
       'pMito', 'patient', 'act_tcell', 'tcell_type', 'activation',
       'mito_filter_out', 'numi_out', 'all_out', 'integrated_snn_res.0.1',
       'integrated_snn_res.0.2', 'integrated_snn_res.0.3',
       'integrated_snn_res.0.4', 'integrated_snn_res.0.5',
       'integrated_snn_res.0.6', 'integrated_snn_res.0.7',
       'integrated_snn_res.0.8', 'integrated_snn_res.0.9',
       'integrated_snn_res.1', 'integrated_snn_res.1.1',
       'integrated_snn_res.1.2', 'seurat_clusters', 'scDblFinder.class',
       'garnett_celltypes', 'activation_status', 'Cell', 'Cluster1',
       'Cluster2', 'down_treg', 'up_treg', 'clusters', 'S.Score', 'G2M.Score',
       'Phase', 'treg_score', 'RNA_snn_res.0.1', 'RNA_snn_res.0.2',
       'RNA_snn_res.0.3', 'RNA_snn_res.0.4', 'RNA_snn_res.0.5',
       'RNA_snn_res.0.6', 'RNA_snn_res.0.7', 'RNA_snn_res.0.8',
       'RNA_snn_res.0.9', 'RNA_snn_res.1', 'RNA_snn_res.1.1',
       

In [22]:
adata_leo_treg.obs = adata_leo_treg.obs[['tcell_type']]
adata_leo_treg.obs.rename(columns={'tcell_type': 'treg'}, inplace=True)
print(adata_leo_treg.obs.columns)
cell_type_counts = adata_leo_treg.obs['treg'].count()
print(cell_type_counts)

Index(['treg'], dtype='object')
48686


combine leo treg and teff sets

In [23]:
adata_leo_combined = sc.concat([adata_leo_treg, adata_leo_teff], merge='same', label=None)


In [24]:
cell_type_counts = adata_leo_combined.obs['treg'].value_counts()
print("\nCell type distribution:")
print(cell_type_counts)
adata_leo_combined.n_obs


Cell type distribution:
treg
Treg    48686
Teff    45955
Name: count, dtype: int64


94641

merge all 3:

In [27]:
#rename
# add prefixes to make indices unique for each dataset
adata_luo.obs_names = [f"Luo_{idx}" for idx in adata_luo.obs_names]
adata_wendering.obs_names = [f"Wendering_{idx}" for idx in adata_wendering.obs_names]
adata_leo_combined.obs_names = [f"Leo_{idx}" for idx in adata_leo_combined.obs_names]


In [29]:
# Identify duplicate indices
print(adata_luo.obs_names.duplicated().sum())
print(adata_wendering.obs_names.duplicated().sum())
print(adata_leo_combined.obs_names.duplicated().sum())

# # Optionally remove duplicates (if any exist)
# adata_luo = adata_luo[~adata_luo.obs_names.duplicated()]
# adata_wendering = adata_wendering[~adata_wendering.obs_names.duplicated()]
# adata_leo_combined = adata_leo_combined[~adata_leo_combined.obs_names.duplicated()]


0
0
0


In [30]:
print(adata_luo.var_names.duplicated().sum())  # Check for duplicates in Luo
print(adata_wendering.var_names.duplicated().sum())  # Check for duplicates in Wendering
print(adata_leo_combined.var_names.duplicated().sum())  # Check for duplicates in Leo


0
10
0


remove 10 wend duplicate genes

In [31]:
adata_wendering = adata_wendering[:, ~adata_wendering.var_names.duplicated()]

In [32]:
print(adata_luo.var_names.duplicated().sum())  # Check for duplicates in Luo
print(adata_wendering.var_names.duplicated().sum())  # Check for duplicates in Wendering
print(adata_leo_combined.var_names.duplicated().sum())  # Check for duplicates in Leo

0
0
0


only common genes:

In [33]:
common_genes = list(
    set(adata_luo.var_names)
    & set(adata_wendering.var_names)
    & set(adata_leo_combined.var_names)
)

adata_luo = adata_luo[:, common_genes]
adata_wendering = adata_wendering[:, common_genes]
adata_leo_combined = adata_leo_combined[:, common_genes]
print(len(common_genes))

2693


In [1]:
print(f"Luo shape: {adata_luo.shape}")
print(f"Wendering shape: {adata_wendering.shape}")
print(f"Leo combined shape: {adata_leo_combined.shape}")


NameError: name 'adata_luo' is not defined

In [34]:
adata_all = sc.concat(
    [adata_luo, adata_wendering, adata_leo_combined],
    label='batch',  # add 'batch' column to track dataset origin
    keys=['Luo', 'Wendering', 'Leo'],  #  batch names
    merge='same'  
)